# 5. praktika: hedapena (bateratu → GGUF → Ollama)

**Colab-en** (koaderno hau): egokitzailea oinarriarekin bateratu, GGUF-ra bihurtu eta `Modelfile` sortu.

**Zure ordenagailuan**: Ollama-n eredua sortu eta `ebaluatu_ollama.py`-rekin ebaluatu.

GPU ez da beharrezkoa koaderno honetan (CPU nahikoa da), baina azkarrago doa.

## 1. Ingurunea

In [ ]:
# COLAB: liburutegiak instalatu (bertsio finkoak, unitate osoan berdinak)
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 datasets==5.2.0

In [ ]:
# COLAB: Google Drive muntatu (emaitzak gordetzeko eta datuak irakurtzeko)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
# Aldatu zure Drive-ko karpetara. Bertan egon behar dute txartelak_train.jsonl eta txartelak_test.jsonl
KARPETA = "/content/drive/MyDrive/llm-egokitzapena"
os.makedirs(KARPETA, exist_ok=True)
TRAIN_FITX = f"{KARPETA}/txartelak_train.jsonl"
TEST_FITX = f"{KARPETA}/txartelak_test.jsonl"

# Fitxategiak Drive-n ez badaude, igo eskuz
if not os.path.exists(TRAIN_FITX):
    from google.colab import files  # COLAB
    igoak = files.upload()
    for izena in igoak:
        os.replace(izena, f"{KARPETA}/{izena}")
print(os.listdir(KARPETA))

In [ ]:
import json
def irakurri_jsonl(fitx):
    with open(fitx, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_lista = irakurri_jsonl(TRAIN_FITX)
test_lista = irakurri_jsonl(TEST_FITX)
SYSTEM = train_lista[0]["prompt"][0]["content"]   # sistema-mezua datuetatik (beti berdina!)
print(len(train_lista), len(test_lista))
print(SYSTEM)

## 2. Egokitzailea bateratu

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODELOA = "Qwen/Qwen2.5-0.5B-Instruct"
EGOKITZAILEA = f"{KARPETA}/txartelak-lora"

base = AutoModelForCausalLM.from_pretrained(MODELOA, dtype=torch.float16)
model = PeftModel.from_pretrained(base, EGOKITZAILEA)
bateratua = model.merge_and_unload()
print(type(model).__name__, "->", type(bateratua).__name__)

bateratua.save_pretrained("txartelak-merged")
tokenizer = AutoTokenizer.from_pretrained(MODELOA)
tokenizer.save_pretrained("txartelak-merged")
!ls -lh txartelak-merged

In [ ]:
# Egiaztatu bateratutako ereduak berdin erantzuten duela
def sailkatu(eredua, testua):
    mezuak = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": testua}]
    sarrera = tokenizer.apply_chat_template(mezuak, add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True)
    irteera = eredua.generate(**sarrera, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(irteera[0][sarrera["input_ids"].shape[1]:], skip_special_tokens=True)

proba = "Idazkaritzako inprimagailuak tinta falta du. Premiazkoa da!"
print(sailkatu(bateratua, proba))

## 3. GGUF-ra bihurtu

`llama.cpp`-ren `convert_hf_to_gguf.py` script-a erabiliko dugu. `q8_0` kuantizazioa: ia jatorrizko kalitatea, tamainaren erdia.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q sentencepiece

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py txartelak-merged --outfile txartelak-q8_0.gguf --outtype q8_0
!ls -lh *.gguf

### (Aukerakoa) `q4_K_M` kuantizazioa

`llama-quantize` tresna konpilatu behar da (5 minutu inguru):

In [ ]:
# !cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF && cmake --build llama.cpp/build --target llama-quantize -j 4
# !python llama.cpp/convert_hf_to_gguf.py txartelak-merged --outfile txartelak-f16.gguf --outtype f16
# !llama.cpp/build/bin/llama-quantize txartelak-f16.gguf txartelak-q4_k_m.gguf Q4_K_M
# !ls -lh *.gguf

## 4. Modelfile sortu

Sistema-mezua **datuetatik** hartzen dugu, entrenamenduko berdin-berdina izan dadin. Txantiloia Qwen-en ChatML formatua da.

In [ ]:
TEMPLATE = """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

modelfile = f'''FROM ./txartelak-q8_0.gguf

TEMPLATE """{TEMPLATE}"""

SYSTEM """{SYSTEM}"""

PARAMETER temperature 0
PARAMETER stop "<|im_end|>"
'''
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile)

## 5. Gorde eta deskargatu

In [ ]:
!mkdir -p {KARPETA}/ollama
!cp txartelak-q8_0.gguf Modelfile {KARPETA}/ollama/
!cp {TEST_FITX} {KARPETA}/ollama/
!ls -lh {KARPETA}/ollama

In [ ]:
# COLAB: zuzenean deskargatu (bestela, Drive-tik)
from google.colab import files
files.download("txartelak-q8_0.gguf")
files.download("Modelfile")

## 6. Zure ordenagailuan (terminala)

Jarri karpeta berean: `txartelak-q8_0.gguf`, `Modelfile`, `txartelak_test.jsonl` eta `ebaluatu_ollama.py`.

```bash
ollama create txartelak -f Modelfile
ollama run txartelak "B05 laborategiko ordenagailua ez da pizten. Premiazkoa!"
ollama list

pip install ollama pandas
python ebaluatu_ollama.py txartelak txartelak_test.jsonl
```

Python-etik:

```python
import json, ollama
r = ollama.chat(model="txartelak", format="json",
                messages=[{"role": "user", "content": "Kaixo, idazkaritzako inprimagailuak tinta falta du."}])
print(json.loads(r.message.content))
```

## 7. Ariketak

1. Bete taula (4. saioko emaitzekin alderatuz):

| Eredua | Guztiz zuzena | Tamaina | s/adibide |
|---|---|---|---|
| LoRA (Colab, fp16) | | ~1 GB | |
| GGUF `q8_0` (Ollama) | | | |
| GGUF `q4_K_M` (aukerakoa) | | | |

2. **Txantiloi okerra**: aldatu `Modelfile`-ko `SYSTEM` (adib. ezabatu) eta sortu `txartelak-txarra` eredua. Ebaluatu. Zer gertatzen da? Zergatik?
3. `format="json"` gabe ebaluatu (`ebaluatu_ollama.py`-n). Aldatzen da `json_ok`?
4. **Erronka**: Streamlit aplikazio txiki bat: testu-kutxa bat, botoi bat eta sailkapena koloretan (lehentasun altua gorriz).
5. **Model card**: idatzi zuen ereduaren `README.md` (ikus `Proiektua.pdf`).